In [ ]:
# Example 1: top-level package metadata + public surface inspection.
import jupyter_server_documents

version = getattr(jupyter_server_documents, '__version__', None)
print(f"version: {version}")

public_names = [n for n in dir(jupyter_server_documents) if not n.startswith('_')]
print(f"public top-level names ({len(public_names)}):")
for n in public_names[:25]:
    print(f"  - {n}")

assert hasattr(jupyter_server_documents, '__path__'), 'package should expose __path__'


In [ ]:
# Example 2: jupyter-server extension entrypoint discovery.
# jupyter_server_documents is a jupyter_server extension; it must expose _jupyter_server_extension_points.
import jupyter_server_documents

ext_points_fn = getattr(jupyter_server_documents, '_jupyter_server_extension_points', None)
print(f"_jupyter_server_extension_points present: {ext_points_fn is not None}")

assert ext_points_fn is not None, 'jupyter_server_documents exposes no _jupyter_server_extension_points'
if ext_points_fn is not None:
    points = ext_points_fn()
    print(f"extension points returned: {points}")
    assert points, 'no extension points declared'
    assert all(p.get('app') is not None for p in points), points
    assert isinstance(points, list), 'extension points should be a list'
    for p in points:
        assert isinstance(p, dict), 'each entry should be a dict'
        assert 'module' in p, 'each entry should declare a module'

print('extension entrypoint contract OK')

In [ ]:
# Example 3: ExtensionApp class discoverable + instantiable.
import jupyter_server_documents
from jupyter_server.extension.application import ExtensionApp

# Find any ExtensionApp subclass declared by the package.
candidates = []
for name in dir(jupyter_server_documents):
    obj = getattr(jupyter_server_documents, name)
    if isinstance(obj, type) and issubclass(obj, ExtensionApp) and obj is not ExtensionApp:
        candidates.append((name, obj))

print(f"ExtensionApp subclasses found: {[n for n, _ in candidates]}")

assert candidates, 'no ExtensionApp subclass at the package top level'
if candidates:
    name, cls = candidates[0]
    inst = cls()
    print(f"instantiated {name}: {inst!r}")
    assert isinstance(inst, ExtensionApp)
else:
    print('no direct ExtensionApp subclass at top level (may live in submodules)')


In [ ]:
# Example 4: pycrdt + jupyter_ydoc integration is reachable.
# jupyter_server_documents depends on these for CRDT-backed collaborative documents.
import jupyter_ydoc
import pycrdt

# Construct a YDoc through pycrdt and exercise a basic op.
doc = pycrdt.Doc()
text = pycrdt.Text()
doc['content'] = text
text += 'hello, collab world'

print(f"pycrdt.Doc: {doc!r}")
print(f"text content: {str(text)!r}")

assert str(text) == 'hello, collab world'
print(f"jupyter_ydoc version: {getattr(jupyter_ydoc, '__version__', 'unknown')}")


In [ ]:
# Example 5: extension is enabled via jupyter_server's extension manager.
from jupyter_server.extension.manager import ExtensionPackage

pkg = ExtensionPackage(name='jupyter_server_documents', enabled=True)
print(f"ExtensionPackage: {pkg!r}")
print(f"  name: {pkg.name}")
print(f"  enabled: {pkg.enabled}")
print(f"  metadata: {pkg.metadata}")

# Each registered point must declare a module name.
for point in pkg.extension_points.values():
    print(f"  point module: {point.module_name}")
    assert point.module_name.startswith('jupyter_server_documents')

assert pkg.name == 'jupyter_server_documents'
assert pkg.enabled is True

import json
from pathlib import Path
from jupyter_core.paths import jupyter_config_path, jupyter_path
_config_files = [Path(d) / 'jupyter_server_config.d' / 'jupyter_server_documents.json' for d in jupyter_config_path()]
_config_files = [f for f in _config_files if f.is_file()]
assert _config_files, f'jupyter_server_documents.json is on no jupyter_config_path() entry: {jupyter_config_path()}'
_flags = [json.loads(f.read_text()).get('ServerApp', {}).get('jpserver_extensions', {}).get('jupyter_server_documents') for f in _config_files]
assert True in _flags and False not in _flags, dict(zip(map(str, _config_files), _flags))
_lab_ext = [Path(d) / '@jupyter-ai-contrib' / 'server-documents' for d in jupyter_path('labextensions')]
_lab_ext = [d for d in _lab_ext if (d / 'package.json').is_file()]
assert _lab_ext, f"@jupyter-ai-contrib/server-documents is on no jupyter_path('labextensions') entry: {jupyter_path('labextensions')}"
assert json.loads((_lab_ext[0] / 'package.json').read_text())['name'] == '@jupyter-ai-contrib/server-documents', _lab_ext[0]
assert list((_lab_ext[0] / 'static').glob('remoteEntry.*.js')), f'no remoteEntry.*.js under {_lab_ext[0]}'
print(f"  enabled by: {_config_files[0]}; labextension: {_lab_ext[0]}")
print('extension successfully registered with jupyter_server')